# Finite Horizons, Terminal Time, and Numerical Bounds

Three separate analyses share the finite 100-need/report, 1001-share scientific engine. Paired repeated-myopic paths compare a 20-question cap with the full affordable horizon. Terminal-time sensitivity evaluates idealized allocation macros at initial beliefs and at the **same old budget-only terminal beliefs**, not a new cost-aware policy distribution. Bellman envelopes concern the **zero-terminal-cost budget-only model**, not the duration-priced model or a transfer to a capped model. None claims globally optimal historical myopic behavior or machine-certified arithmetic.

Expected utility (or its affine combined-GPA transform) is the performance measure. Question counts, shares and matching rates are descriptive diagnostics.

In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the cloned repository.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])

## Settings and Scope
The full paired recipe has root seed 2026100901, episode IDs 0..399 in each of six C3/C5 templates, and independent latent/report seed namespaces for each recipient. Integer-first resource arithmetic, half-step allocation choices and decimal display rounding live in source, not this notebook. The input design retains the capped baseline; `budget_templates` explicitly derives the uncapped affordable horizon for these diagnostics.

Timing profiles are assumptions, not measured human durations. The reference rate is the symmetric C1 scale-1 native-optimal GPA gain from 120 to 121 tutoring minutes per personal second. Personal question time is not priced; terminal personal time never deducts tutoring resource or task GPA. Zero resource has zero terminal duration. Expected-GPA equal outcome and raw-surplus equal outcome are different macros.

Root refinement defaults are 600 seconds, 100000 created states and depth 6. These limit computation, **not legal questioning**. Full-affordable-horizon upper envelopes and feasible stop-at-frontier witnesses remain in place. Hardware-dependent cutoff timing means final bound endpoints/DAGs need not be bit-identical. Atomic expansions can exceed an elapsed-time cutoff.

In [ ]:
import json, os, shlex, subprocess, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
RUN = False
MODE = 'full'  # 'smoke' runs one paired terminal and an unexpanded root only.
OUTPUT = ROOT / 'results' / 'finite_horizon_terminal_bounds'
DESIGN = ROOT / 'configs' / 'tutoring_conditions.json'
PAIRED_SETTINGS = json.loads((ROOT / 'configs' / 'tutoring_paired_horizon.json').read_text())
TIME_SETTINGS = json.loads((ROOT / 'configs' / 'tutoring_terminal_time_v01.json').read_text())
BOUND_SETTINGS = json.loads((ROOT / 'configs' / 'tutoring_optimality_v01.json').read_text())
ROOT_SEED = PAIRED_SETTINGS['root_seed']
EPISODES_PER_TEMPLATE = PAIRED_SETTINGS['episodes_per_active_template']
ROOT_LIMITS = dict(BOUND_SETTINGS['production'])
ACTIVE_TEMPLATES = [f'{condition}_s{scale}_g0' for condition in ('C3', 'C5') for scale in range(3)]
design = json.loads(DESIGN.read_text())
print('Design: scales, grid, and utility:', design['common']['variation']['scale_factors'], design['common']['allocation']['share_grid_denominator'], design['common']['utility'])
print('Condition priors, report widths, resources and question costs:', design['conditions'])
print('Paired seed/episodes:', ROOT_SEED, EPISODES_PER_TEMPLATE)
print('C1 gaps:', TIME_SETTINGS['C1_total_gaps_minutes'], 'rho multipliers:', TIME_SETTINGS['rho_multipliers'])
print('Timing profiles:', TIME_SETTINGS['assumed_profiles'])
print('Independent EO/OPT seconds:', TIME_SETTINGS['C1_independent_inference_seconds'])
print('Bound limits/guard/margin:', ROOT_LIMITS, BOUND_SETTINGS['numerical_guard'], BOUND_SETTINGS['certificate_margin'])
print('Execution enabled:', RUN)

## Regenerate Inputs and Analyses
All scientific inputs are generated from the public design and settings. The paired producer writes `pairs.jsonl`, scientific template/seed metadata and the source summary. Both terminal consumers replay that same supplied panel and check posterior/resource parity. Output directories must be new. Full initial sensitivity includes 36 states and 1800 scenario rows; the 2400 old terminals each produce 18 sensitivity scenarios. Smoke mode is only a small interface check and omits initial sensitivity. Root checkpoints alternate complete DAG witnesses every 30 seconds between atomic operations and at final return; the checkpoint pointer identifies the matching witness.

These commands do **not** reproduce the 45 ENGAGEMENT schedules, 6804 balanced geometry cases, JavaScript source-parity panel, conditional power calculations, or separate derived equal-template/working-stimulus aggregation tables. The raw initial/scenario records, orientation audit, break-even quantities and terminal macro counts are provided. No full cost-aware policy calculation or human inference/calibration is included.

In [ ]:
if MODE not in ('full', 'smoke'):
    raise ValueError('Choose full or smoke.')
runner = [sys.executable, str(ROOT / 'scripts' / 'reproduce_finite_tutoring.py')]
settings_dir = OUTPUT / 'scientific_settings'
paired_config = settings_dir / 'paired.json'
time_config = settings_dir / 'terminal_time.json'
bound_config = settings_dir / 'bounds.json'
pairs = OUTPUT / 'pairs'
def command(stage, output, config, *options):
    return runner + [stage, '--design', str(DESIGN), '--config', str(config), '--output', str(output), *map(str, options)]
pair_options = ['--episodes', EPISODES_PER_TEMPLATE] if MODE == 'full' else ['--episodes', 1, '--template', ACTIVE_TEMPLATES[0]]
commands = [command('pairs', pairs, paired_config, *pair_options)]
if MODE == 'full':
    commands += [command('horizon-initial', OUTPUT / 'horizon_initial', paired_config),
                 command('time-initial', OUTPUT / 'time_initial', time_config),
                 command('bounds-initial', OUTPUT / 'bounds_initial', bound_config)]
commands += [command('time-terminal', OUTPUT / 'time_terminal', time_config, '--pairs', pairs),
             command('bounds-terminal', OUTPUT / 'bounds_terminal', bound_config, '--pairs', pairs)]
templates = ACTIVE_TEMPLATES if MODE == 'full' else ACTIVE_TEMPLATES[:1]
limits = ROOT_LIMITS if MODE == 'full' else {'max_seconds': 0, 'max_created_states': 1, 'max_depth': 0}
for template in templates:
    commands.append(command('bounds-root', OUTPUT / 'roots' / template, bound_config, '--template', template,
                            '--max-seconds', limits['max_seconds'], '--max-created-states', limits['max_created_states'], '--max-depth', limits['max_depth']))
for cmd in commands:
    print(shlex.join(cmd))
if RUN:
    settings_dir.mkdir(parents=True, exist_ok=False)
    for path, settings in [(paired_config, {**PAIRED_SETTINGS, 'root_seed': ROOT_SEED}), (time_config, TIME_SETTINGS), (bound_config, BOUND_SETTINGS)]:
        path.write_text(json.dumps(settings, indent=2, allow_nan=False) + '\n')
    env = {**os.environ, 'OPENBLAS_NUM_THREADS': '1', 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}
    for cmd in commands:
        subprocess.run(cmd, cwd=ROOT, env=env, check=True)

In [ ]:
if RUN:
    paired_summary = json.loads((pairs / 'summary.json').read_text())
    for row in paired_summary:
        print(row['label'], 'pairs:', row['pairs'], 'utility difference:', row['paired_full_minus_capped_utility'], 'different paths:', row['different_event_paths'])
    for name in ('time_terminal', 'bounds_terminal'):
        print(name, json.loads((OUTPUT / name / 'summary.json').read_text()))
    for template in templates:
        row = json.loads((OUTPUT / 'roots' / template / 'result.json').read_text())
        print(template, {key: row[key] for key in ('lower', 'upper', 'gap', 'action_status', 'limit_reason', 'scientific_horizon', 'limits', 'witness_file')})
else:
    print('No computation or output readback requested. Inspect settings, then explicitly enable RUN.')